In [ ]:
from __future__ import annotations

import hashlib
import math
import os
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.model_selection import StratifiedKFold, train_test_split
from tensorflow.keras import Model
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.utils import Sequence as KerasSequence, to_categorical

In [ ]:
# Reproducibility

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
# Labels and config

TMD_LABELS = ["normal", "subluxation"]
DISPLAY_LABELS = ["Normal", "Subluxation"]
CLASS_NAME_ALIASES = {
    "normal": 0,
    "subluxation": 1,
    "sl": 1,
}
ARTIFACT_LABELS = ["none", "motion_blur", "gaussian_noise", "metal_streak"]
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

@dataclass
class ExperimentConfig:
    dataset_root: str
    image_size: Tuple[int, int] = (224, 224)
    batch_size: int = 8
    epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay: float = 1e-2
    n_splits: int = 5
    val_size: float = 0.15
    random_state: int = 42
    freeze_backbone: bool = False
    benchmark_attention: str = "self"  # corrected baseline attention
    proposed_attention: str = "cbam"
    artifact_mix_probability: Tuple[float, float, float, float] = (0.25, 0.25, 0.25, 0.25)
    tmd_loss_weight: float = 1.0
    artifact_loss_weight: float = 0.35
    num_workers: int = 1
    max_queue_size: int = 10

In [ ]:
# Dataset indexing

def index_dataset(dataset_root: str) -> pd.DataFrame:
    root = Path(dataset_root)
    if not root.exists():
        raise FileNotFoundError(f"Dataset root does not exist: {dataset_root}")

    rows: List[Dict[str, object]] = []
    class_to_idx = {name: idx for idx, name in enumerate(TMD_LABELS)}
    split_names = ["train", "validation", "test"]

    # If split folders exist, POOL all images across them for true 5-fold CV.
    has_split_layout = all((root / split_name).exists() for split_name in split_names)

    if has_split_layout:
        for split_name in split_names:
            split_dir = root / split_name
            for class_name in TMD_LABELS:
                class_dir = split_dir / class_name
                if not class_dir.exists():
                    raise FileNotFoundError(
                        f"Expected class folder '{class_name}' inside {split_dir}"
                    )

                for file_path in class_dir.rglob("*"):
                    if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                        rows.append(
                            {
                                "filepath": str(file_path),
                                "tmd_label": class_to_idx[class_name],
                                "class_name": class_name,
                                "original_split": split_name,
                            }
                        )
    else:
        # Flat layout also supported: dataset_root/normal and dataset_root/subluxation
        for class_name in TMD_LABELS:
            class_dir = root / class_name
            if not class_dir.exists():
                raise FileNotFoundError(
                    f"Expected class folder '{class_name}' inside {dataset_root}"
                )
            for file_path in class_dir.rglob("*"):
                if file_path.is_file() and file_path.suffix.lower() in IMAGE_EXTENSIONS:
                    rows.append(
                        {
                            "filepath": str(file_path),
                            "tmd_label": class_to_idx[class_name],
                            "class_name": class_name,
                            "original_split": "pooled",
                        }
                    )

    if not rows:
        raise ValueError(
            "No image files found. Expected either:\n"
            "1) dataset_root/train|validation|test/normal|subluxation\n"
            "or\n"
            "2) dataset_root/normal|subluxation"
        )

    df = pd.DataFrame(rows).sort_values("filepath").reset_index(drop=True)
    return df


In [ ]:
# Image corruption utilities

def _ensure_uint8(image: np.ndarray) -> np.ndarray:
    image = np.clip(image, 0, 255)
    return image.astype(np.uint8)


def add_motion_blur(image: np.ndarray, kernel_size: int = 9) -> np.ndarray:
    kernel_size = max(3, int(kernel_size))
    if kernel_size % 2 == 0:
        kernel_size += 1
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    kernel[kernel_size // 2, :] = 1.0
    kernel /= kernel.sum()
    blurred = cv2.filter2D(image, -1, kernel)
    return _ensure_uint8(blurred)


def add_gaussian_noise(image: np.ndarray, sigma: float = 18.0) -> np.ndarray:
    noise = np.random.normal(0, sigma, image.shape)
    noisy = image.astype(np.float32) + noise
    return _ensure_uint8(noisy)


def add_metal_streak(image: np.ndarray, num_streaks: int = 3) -> np.ndarray:
    h, w = image.shape[:2]
    output = image.copy().astype(np.float32)

    for _ in range(max(1, num_streaks)):
        x1 = np.random.randint(0, w)
        y1 = np.random.randint(0, h)
        x2 = np.random.randint(0, w)
        y2 = np.random.randint(0, h)
        thickness = np.random.randint(2, 8)

        overlay = np.zeros_like(output)
        intensity = np.random.randint(180, 255)
        cv2.line(overlay, (x1, y1), (x2, y2), (intensity, intensity, intensity), thickness)
        overlay = cv2.GaussianBlur(overlay, (0, 0), sigmaX=2.0)
        output = np.maximum(output, overlay)

    return _ensure_uint8(output)


def apply_artifact(image: np.ndarray, artifact_label: int) -> np.ndarray:
    if artifact_label == 0:
        return image
    if artifact_label == 1:
        return add_motion_blur(image)
    if artifact_label == 2:
        return add_gaussian_noise(image)
    if artifact_label == 3:
        return add_metal_streak(image)
    raise ValueError(f"Unknown artifact label: {artifact_label}")

In [ ]:
# Data generator

class TMJSequence(KerasSequence):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        image_size: Tuple[int, int],
        batch_size: int,
        multi_task: bool,
        scenario: str,
        training: bool,
        seed: int = 42,
    ) -> None:
        self.df = dataframe.reset_index(drop=True).copy()
        self.image_size = image_size
        self.batch_size = batch_size
        self.multi_task = multi_task
        self.scenario = scenario
        self.training = training
        self.seed = seed
        self.indices = np.arange(len(self.df))
        self.rng = np.random.default_rng(seed)
        self.on_epoch_end()

    def __len__(self) -> int:
        return math.ceil(len(self.df) / self.batch_size)

    def on_epoch_end(self) -> None:
        if self.training:
            self.rng.shuffle(self.indices)

    def _deterministic_artifact(self, filepath: str) -> int:
        if self.scenario == "clean":
            return 0
        digest = hashlib.md5(filepath.encode("utf-8")).hexdigest()
        return int(digest, 16) % len(ARTIFACT_LABELS)

    def _sample_artifact(self, filepath: str) -> int:
        if self.scenario == "clean":
            return 0
        if self.training:
            return int(self.rng.choice(np.arange(len(ARTIFACT_LABELS))))
        return self._deterministic_artifact(filepath)

    def _load_image(self, filepath: str) -> np.ndarray:
        image = cv2.imread(filepath)
        if image is None:
            raise ValueError(f"Failed to load image: {filepath}")
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, self.image_size, interpolation=cv2.INTER_AREA)
        return image

    def _augment_base(self, image: np.ndarray) -> np.ndarray:
        if self.training and self.rng.random() < 0.5:
            image = cv2.flip(image, 1)
        return image

    def __getitem__(self, index: int):
        batch_indices = self.indices[index * self.batch_size : (index + 1) * self.batch_size]
        batch_df = self.df.iloc[batch_indices]

        images: List[np.ndarray] = []
        tmd_labels: List[int] = []
        artifact_labels: List[int] = []

        for row in batch_df.itertuples(index=False):
            image = self._load_image(row.filepath)
            image = self._augment_base(image)
            artifact_label = self._sample_artifact(row.filepath)
            image = apply_artifact(image, artifact_label)
            image = image.astype(np.float32) / 255.0

            images.append(image)
            tmd_labels.append(int(row.tmd_label))
            artifact_labels.append(int(artifact_label))

        x = np.stack(images, axis=0)
        y_tmd = to_categorical(np.array(tmd_labels), num_classes=len(TMD_LABELS))
        y_artifact = to_categorical(np.array(artifact_labels), num_classes=len(ARTIFACT_LABELS))

        if self.multi_task:
            return x, {"tmd_output": y_tmd, "artifact_output": y_artifact}
        return x, y_tmd

In [ ]:
# Attention blocks

class AttentionBlock(layers.Layer):
    def __init__(self, attention_type: str = "cbam", reduction_ratio: int = 16, **kwargs):
        super().__init__(**kwargs)
        self.attention_type = attention_type.lower()
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        filters = int(input_shape[-1])
        reduced_filters = max(filters // self.reduction_ratio, 1)

        if self.attention_type == "self":
            qk_filters = max(filters // 8, 1)
            self.query_conv = layers.Conv2D(qk_filters, 1, padding="same")
            self.key_conv = layers.Conv2D(qk_filters, 1, padding="same")
            self.value_conv = layers.Conv2D(filters, 1, padding="same")
        elif self.attention_type == "cbam":
            self.avg_pool = layers.GlobalAveragePooling2D()
            self.max_pool = layers.GlobalMaxPooling2D()
            self.shared_dense_1 = layers.Dense(reduced_filters, activation="relu")
            self.shared_dense_2 = layers.Dense(filters)
            self.spatial_conv = layers.Conv2D(1, 7, padding="same", activation="sigmoid")
        else:
            raise ValueError(f"Unsupported attention type: {self.attention_type}")

        super().build(input_shape)

    def call(self, inputs):
        if self.attention_type == "self":
            shape = tf.shape(inputs)
            batch_size, height, width = shape[0], shape[1], shape[2]
            channels = inputs.shape[-1]

            q = self.query_conv(inputs)
            k = self.key_conv(inputs)
            v = self.value_conv(inputs)

            q = tf.reshape(q, [batch_size, height * width, q.shape[-1]])
            k = tf.reshape(k, [batch_size, height * width, k.shape[-1]])
            v = tf.reshape(v, [batch_size, height * width, channels])

            attention_scores = tf.matmul(q, k, transpose_b=True)
            attention_scores = attention_scores / tf.math.sqrt(tf.cast(tf.shape(k)[-1], tf.float32))
            attention_weights = tf.nn.softmax(attention_scores, axis=-1)
            attended = tf.matmul(attention_weights, v)
            attended = tf.reshape(attended, [batch_size, height, width, channels])
            return inputs + attended

        avg_descriptor = self.avg_pool(inputs)
        max_descriptor = self.max_pool(inputs)
        avg_descriptor = self.shared_dense_2(self.shared_dense_1(avg_descriptor))
        max_descriptor = self.shared_dense_2(self.shared_dense_1(max_descriptor))
        channel_attention = tf.nn.sigmoid(avg_descriptor + max_descriptor)
        channel_attention = tf.reshape(channel_attention, (-1, 1, 1, inputs.shape[-1]))
        x = inputs * channel_attention

        avg_map = tf.reduce_mean(x, axis=-1, keepdims=True)
        max_map = tf.reduce_max(x, axis=-1, keepdims=True)
        spatial_attention = self.spatial_conv(tf.concat([avg_map, max_map], axis=-1))
        return x * spatial_attention

In [ ]:
# Model builders

def _make_backbone(image_size: Tuple[int, int], trainable: bool) -> Model:
    backbone = tf.keras.applications.DenseNet201(
        include_top=False,
        weights="imagenet",
        input_shape=(*image_size, 3),
        pooling=None,
    )
    backbone.trainable = trainable
    return backbone


def build_corrected_benchmark_model(config: ExperimentConfig) -> Model:
    """
    Corrected benchmark version.

    The uploaded baseline computes self-attention on pool3 features but then discards it.
    This implementation actually fuses the attention branch into the forward path.
    """
    backbone = _make_backbone(config.image_size, trainable=not config.freeze_backbone)

    pool3 = backbone.get_layer("pool3_relu").output
    pool3_att = AttentionBlock(attention_type="self", name="benchmark_self_attention")(pool3)
    pool3_down = layers.AveragePooling2D(pool_size=2, name="benchmark_pool3_downsample")(pool3_att)

    conv5 = backbone.get_layer("conv5_block32_concat").output
    pool3_proj = layers.Conv2D(
        filters=int(conv5.shape[-1]), kernel_size=1, padding="same", name="benchmark_pool3_projection"
    )(pool3_down)

    fused = layers.Concatenate(name="benchmark_fused_features")([conv5, pool3_proj])
    fused = layers.Conv2D(1024, kernel_size=1, activation="relu", padding="same", name="benchmark_fusion_conv")(fused)
    gap = layers.GlobalAveragePooling2D(name="benchmark_gap")(fused)
    x = layers.Dense(512, activation="relu", kernel_regularizer=l2(config.weight_decay), name="benchmark_fc1")(gap)
    x = layers.Dropout(0.5, name="benchmark_dropout")(x)
    x = layers.BatchNormalization(name="benchmark_bn")(x)
    x = layers.Dense(128, activation="relu", name="benchmark_fc2")(x)
    outputs = layers.Dense(len(TMD_LABELS), activation="softmax", name="tmd_output")(x)

    return Model(inputs=backbone.input, outputs=outputs, name="DenseNet201_Benchmark_Corrected")



def build_agml_model(config: ExperimentConfig) -> Model:
    """
    Proposed thesis model:
    DenseNet201 backbone + CBAM + shared representation + dual heads.
    """
    backbone = _make_backbone(config.image_size, trainable=not config.freeze_backbone)

    conv5 = backbone.get_layer("conv5_block32_concat").output
    attended = AttentionBlock(attention_type="cbam", name="cbam_attention")(conv5)
    attended = layers.Conv2D(1024, kernel_size=1, activation="relu", padding="same", name="cbam_refine_conv")(attended)

    shared = layers.GlobalAveragePooling2D(name="shared_gap")(attended)
    shared = layers.Dense(512, activation="relu", kernel_regularizer=l2(config.weight_decay), name="shared_fc1")(shared)
    shared = layers.Dropout(0.5, name="shared_dropout")(shared)
    shared = layers.BatchNormalization(name="shared_bn")(shared)
    shared = layers.Dense(128, activation="relu", name="shared_fc2")(shared)

    tmd_output = layers.Dense(len(TMD_LABELS), activation="softmax", name="tmd_output")(shared)
    artifact_output = layers.Dense(
        len(ARTIFACT_LABELS), activation="softmax", name="artifact_output"
    )(shared)

    return Model(inputs=backbone.input, outputs=[tmd_output, artifact_output], name="DenseNet201_CBAM_AGML")


In [ ]:
# Training helpers

def make_callbacks() -> List[tf.keras.callbacks.Callback]:
    return [
        EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ]



def compile_model(model: Model, multi_task: bool, config: ExperimentConfig) -> None:
    if multi_task:
        model.compile(
            optimizer=Adam(config.learning_rate),
            loss={
                "tmd_output": "categorical_crossentropy",
                "artifact_output": "categorical_crossentropy",
            },
            loss_weights={
                "tmd_output": config.tmd_loss_weight,
                "artifact_output": config.artifact_loss_weight,
            },
            metrics={
                "tmd_output": ["accuracy"],
                "artifact_output": ["accuracy"],
            },
        )
    else:
        model.compile(
            optimizer=Adam(config.learning_rate),
            loss="categorical_crossentropy",
            metrics=["accuracy"],
        )



def split_train_val(train_df: pd.DataFrame, val_size: float, random_state: int) -> Tuple[pd.DataFrame, pd.DataFrame]:
    train_part, val_part = train_test_split(
        train_df,
        test_size=val_size,
        stratify=train_df["tmd_label"],
        random_state=random_state,
    )
    return train_part.reset_index(drop=True), val_part.reset_index(drop=True)

In [ ]:
# Metrics and evaluation

def compute_binary_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0
    )
    accuracy = (tp + tn) / max(cm.sum(), 1)
    specificity = tn / max(tn + fp, 1)
    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "specificity": float(specificity),
        "f1": float(f1),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

def _collect_predictions(model: Model, generator: TMJSequence, multi_task: bool) -> Tuple[np.ndarray, np.ndarray]:
    y_true: List[int] = []
    y_pred: List[int] = []

    for batch_x, batch_y in generator:
        preds = model.predict(batch_x, verbose=0)
        if multi_task:
            pred_tmd = preds[0]
            true_tmd = batch_y["tmd_output"]
        else:
            pred_tmd = preds
            true_tmd = batch_y

        y_true.extend(np.argmax(true_tmd, axis=1).tolist())
        y_pred.extend(np.argmax(pred_tmd, axis=1).tolist())

    return np.array(y_true), np.array(y_pred)



def measure_inference_speed(model: Model, generator: TMJSequence, warmup_batches: int = 1) -> float:
    batches = list(range(len(generator)))
    if not batches:
        raise ValueError("Generator is empty; cannot measure inference speed.")

    for i in range(min(warmup_batches, len(generator))):
        batch_x, _ = generator[i]
        model.predict(batch_x, verbose=0)

    total_images = 0
    start = time.perf_counter()
    for i in batches:
        batch_x, _ = generator[i]
        _ = model.predict(batch_x, verbose=0)
        total_images += len(batch_x)
    elapsed = time.perf_counter() - start

    return total_images / max(elapsed, 1e-8)



def summarize_cv_results(results_df: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "accuracy",
        "precision",
        "recall",
        "specificity",
        "f1",
        "images_per_second",
    ]
    summary = results_df[metric_cols].agg(["mean", "std"]).T
    summary.columns = ["mean", "std"]
    return summary.reset_index(names="metric")

In [ ]:
def run_cross_validation(
    config: ExperimentConfig,
    model_type: str = "proposed",
    scenario: str = "clean",
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    model_type: 'benchmark' or 'proposed'
    scenario: 'clean' or 'artifact_mix'
    """
    if model_type not in {"benchmark", "proposed"}:
        raise ValueError("model_type must be 'benchmark' or 'proposed'")
    if scenario not in {"clean", "artifact_mix"}:
        raise ValueError("scenario must be 'clean' or 'artifact_mix'")

    seed_everything(config.random_state)
    df = index_dataset(config.dataset_root)
    skf = StratifiedKFold(
        n_splits=config.n_splits,
        shuffle=True,
        random_state=config.random_state
    )

    fold_rows: List[Dict[str, float]] = []
    confusion_matrices: List[np.ndarray] = []

    for fold_idx, (train_val_idx, test_idx) in enumerate(
        skf.split(df["filepath"], df["tmd_label"]),
        start=1
    ):
        train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
        test_df = df.iloc[test_idx].reset_index(drop=True)

        train_df, val_df = split_train_val(
            train_val_df,
            config.val_size,
            config.random_state + fold_idx
        )

        multi_task = model_type == "proposed"

        train_gen = TMJSequence(
            train_df,
            image_size=config.image_size,
            batch_size=config.batch_size,
            multi_task=multi_task,
            scenario=scenario,
            training=True,
            seed=config.random_state + fold_idx,
        )
        val_gen = TMJSequence(
            val_df,
            image_size=config.image_size,
            batch_size=config.batch_size,
            multi_task=multi_task,
            scenario=scenario,
            training=False,
            seed=config.random_state + fold_idx,
        )
        test_gen = TMJSequence(
            test_df,
            image_size=config.image_size,
            batch_size=config.batch_size,
            multi_task=multi_task,
            scenario=scenario,
            training=False,
            seed=config.random_state + fold_idx,
        )

        if model_type == "benchmark":
            model = build_corrected_benchmark_model(config)
        else:
            model = build_agml_model(config)

        compile_model(model, multi_task=multi_task, config=config)

        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=config.epochs,
            callbacks=make_callbacks(),
            verbose=1,
        )

        y_true, y_pred = _collect_predictions(model, test_gen, multi_task=multi_task)
        metrics = compute_binary_metrics(y_true, y_pred)
        images_per_second = measure_inference_speed(model, test_gen)
        confusion_matrices.append(confusion_matrix(y_true, y_pred, labels=[0, 1]))

        fold_rows.append(
            {
                "fold": fold_idx,
                "split_mode": "kfold",
                "model_type": model_type,
                "scenario": scenario,
                **metrics,
                "images_per_second": float(images_per_second),
                "epochs_ran": len(history.history.get("loss", [])),
            }
        )

        tf.keras.backend.clear_session()

    results_df = pd.DataFrame(fold_rows)
    pooled_cm = np.sum(confusion_matrices, axis=0)
    pooled_cm_df = pd.DataFrame(pooled_cm, index=DISPLAY_LABELS, columns=DISPLAY_LABELS)
    return results_df, pooled_cm_df


In [ ]:
# Grad-CAM

def _find_target_layer(model: Model, preferred: Optional[str] = None) -> str:
    if preferred is not None:
        return preferred
    for candidate in ["cbam_refine_conv", "benchmark_fusion_conv", "conv5_block32_concat"]:
        try:
            model.get_layer(candidate)
            return candidate
        except ValueError:
            continue
    raise ValueError("No suitable Grad-CAM target layer found.")



def make_gradcam_heatmap(
    model: Model,
    image_array: np.ndarray,
    target_class: Optional[int] = None,
    target_layer: Optional[str] = None,
    output_name: str = "tmd_output",
) -> np.ndarray:
    target_layer = _find_target_layer(model, target_layer)

    if image_array.ndim == 3:
        image_batch = np.expand_dims(image_array, axis=0)
    else:
        image_batch = image_array

    conv_layer = model.get_layer(target_layer)
    grad_model = Model(
        inputs=model.inputs,
        outputs=[conv_layer.output, model.get_layer(output_name).output],
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(image_batch)
        if target_class is None:
            target_class = int(tf.argmax(predictions[0]))
        class_channel = predictions[:, target_class]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)
    heatmap = tf.maximum(heatmap, 0) / tf.maximum(tf.reduce_max(heatmap), 1e-8)
    return heatmap.numpy()



def overlay_gradcam(
    rgb_image: np.ndarray,
    heatmap: np.ndarray,
    alpha: float = 0.35,
) -> np.ndarray:
    image = rgb_image.copy()
    if image.max() <= 1.0:
        image = (image * 255).astype(np.uint8)
    heatmap = cv2.resize(heatmap, (image.shape[1], image.shape[0]))
    heatmap_uint8 = np.uint8(255 * heatmap)
    color_map = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)
    color_map = cv2.cvtColor(color_map, cv2.COLOR_BGR2RGB)
    blended = cv2.addWeighted(image, 1.0 - alpha, color_map, alpha, 0)
    return blended

In [ ]:
# Plotting helpers

def plot_confusion_matrix(cm_df: pd.DataFrame, title: str = "Confusion Matrix") -> None:
    plt.figure(figsize=(6, 5))
    plt.imshow(cm_df.values, cmap="Blues")
    plt.title(title)
    plt.xticks(range(len(cm_df.columns)), cm_df.columns)
    plt.yticks(range(len(cm_df.index)), cm_df.index)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    for i in range(cm_df.shape[0]):
        for j in range(cm_df.shape[1]):
            plt.text(j, i, int(cm_df.iloc[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.show()

In [ ]:
# Run this cell after all definitions above

cfg = ExperimentConfig(
    dataset_root="data",   # points to data/train, data/validation, data/test
    image_size=(224, 224),
    batch_size=8,
    epochs=30,
    learning_rate=1e-4,
)

# Your current folder structure is a fixed split, so use this:
results, cm = run_fixed_split_experiment(
    cfg,
    model_type="proposed",     # "proposed" or "benchmark"
    scenario="artifact_mix"    # "clean" or "artifact_mix"
)

print(results)
print(cm)
plot_confusion_matrix(cm, title="Proposed Model - Artifact Mix")


In [ ]:
# True 5-fold run using your existing data/train, data/validation, data/test as a pooled source

cfg = ExperimentConfig(
    dataset_root="data",
    image_size=(224, 224),
    batch_size=8,
    epochs=30,
    learning_rate=1e-4,
    n_splits=5,
    val_size=0.15,
)

results_cv, cm_cv = run_cross_validation(
    cfg,
    model_type="proposed",      # or "benchmark"
    scenario="artifact_mix"     # or "clean"
)

print(results_cv)
print()
print("Mean metrics across folds:")
print(results_cv[["accuracy", "precision", "recall", "specificity", "f1", "images_per_second"]].mean())
print()
print(cm_cv)
plot_confusion_matrix(cm_cv.values, title="Proposed Model - Artifact Mix - 5 Fold CV")
